# Customer churn 

Customer churn is a problem that affects all industries.
Often a lot of energy is put into acquiring customers, but retaining them is neglected.
With a customer churn forecast, those customers who are likely to churn can be identified.
The company can then perhaps win them back with special offers or other attention.
Data pre-processing for classification is discussed as part of the exercise. 
The decision tree as a classifier is repeated and deepened from the first half of the semester. 
At the end of the exercise, you will be able to carry out a quality assessment of the model for typical classification problems and evaluate an existing ML solution in terms of its suitability for the problem.

The Jupyter Notebook depends on the scikit-learn version, among other things.
Please make sure that your version is up to date:

In [1]:
import sklearn
sklearn.__version__

'1.7.2'

An overview of the current version can be found at https://scikit-learn.org/dev/versions.html.
This notebook is designed for version 1.2.

## Task 1: Creating the basic framework

### Task 1.1: Importing modules

<span style="color:blue">
 As a first step, load the libraries for analyzing and manipulating data presented in the lecture and first exercise, as well as scikit-learn.
</span>

In [2]:
import scipy

Now we download the file `CustomerChurn.xlsx`.
The code for this is already provided.
IBM presents the data set
[in a blog post](https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113)
before.

In [5]:
import os
import requests
import shutil
import numpy as np
import pandas as pd

file_name = "CustomerChurn.xlsx"
if file_name not in os.listdir("."):
    print(f"Download file '{file_name}'")
    r = requests.get("https://public.dhe.ibm.com/software/data/sw-library/cognos/mobile/C11/data/CustomerChurn.xlsx", stream=True)
    if r.status_code == 200:
        with open(file_name, 'wb') as f:
            r.raw.decode_content = True
            shutil.copyfileobj(r.raw, f)
else:
   print(f"File '{file_name}' is already downloaded")

print("Contents of folder:")
for file_entry in os.listdir("."):
    print("- ", file_entry)

File 'CustomerChurn.xlsx' is already downloaded
Contents of folder:
-  02 Exercise Customer Churn.ipynb
-  CustomerChurn.xlsx


#### Task 1.2: Reading the data set

In task 1.2, the data required for the analyses should be read in.
The module `pandas` provides the option of reading Excel files with the function `read_excel()`. 
You can find more information at the following link:
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_excel.html

<span style="color:blue">Load the file `CustomerChurn.xlsx` using the procedure shown in the previous exercise.
 Save the data in a DataFrame called `df` and have it visualized.
</span> 

In [9]:
df = pd.read_excel(file_name)
df.head() 

,LoyaltyID,Customer ID,Senior Citizen,Partner,Dependents,Tenure,Phone Service,Multiple Lines,Internet Service,Online Security,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn
0,318537,7590-VHVEG,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,152148,5575-GNVDE,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,326527,3668-QPYBK,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,845894,7795-CFOCW,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,503388,9237-HQITU,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


#### Task 1.3: Output of individual columns

By specifying the column name, various data from the dataframe can be output.
As an example, the following statement outputs the rows of the columns `Senior Citizen` and `Paperless Billing`.

```python
df[['Senior Citizen', 'Paperless Billing']]
```

<span style="color:blue">Output the rows of the `Total Charges`, `Churn` and `Online Security` columns of the DataFrame.
</span> 

In [11]:
df[['Total Charges', 'Churn', 'Online Security']].head()

,Total Charges,Churn,Online Security
0,29.85,No,No
1,1889.5,No,Yes
2,108.15,Yes,Yes
3,1840.75,No,Yes
4,151.65,Yes,No


## Task 2: Data preprocessing <a class="anchor" id="second-task"></a>

In task 2, the data required for the analyses should be preprocessed.

#### Task 2.1: Determining the scales of the columns

In the lecture, various scales for attributes were presented.
The method `df.info()` shows which type has been determined within pandas.
A short technical explanation of the types is available
[in Manual](https://pandas.pydata.org/docs/user_guide/basics.html#dtypes).

<span style="color:blue">Execute the following statement and interpret the result.
 Which attribute is currently being measured on which scale?
 Which scale should the respective attribute actually have?
</span>

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   LoyaltyID          7043 non-null   int64  
 1   Customer ID        7043 non-null   object 
 2   Senior Citizen     7043 non-null   object 
 3   Partner            7043 non-null   object 
 4   Dependents         7043 non-null   object 
 5   Tenure             7043 non-null   int64  
 6   Phone Service      7043 non-null   object 
 7   Multiple Lines     7043 non-null   object 
 8   Internet Service   7043 non-null   object 
 9   Online Security    7043 non-null   object 
 10  Online Backup      7043 non-null   object 
 11  Device Protection  7043 non-null   object 
 12  Tech Support       7043 non-null   object 
 13  Streaming TV       7043 non-null   object 
 14  Streaming Movies   7043 non-null   object 
 15  Contract           7043 non-null   object 
 16  Paperless Billing  7043 

In [ ]:
from sklearn import model_selection
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

In this exercise, a decision tree is used to classify whether a customer is about to leave the company or not.
To create this tree, an instance of an untrained decision tree classifier is first initialized
(Here, the maximum depth of the tree is initially set to 3).

In [ ]:
from sklearn import tree
clf_dt = tree.DecisionTreeClassifier(max_depth=3)

In a next step, the decision tree is trained using the training data using the ```fit()``` method (http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.fit).

In [ ]:
clf_dt.fit(X_train, y_train)

The scikit-learn decision tree not only needs numerical values ​​everywhere, but also no data can be missing.
That's why we simply discard all rows in which we have missing values.

In [ ]:
df.dropna(axis=1, how='any', inplace=True)
df

Now we call the learning process again:

In [ ]:
X = df.drop(['Customer ID', 'LoyaltyID', 'Churn'], axis=1).values
y = df['Churn'].values
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

clf_dt.fit(X_train, y_train)

Now that the decision tree is ready, its quality can be assessed using the test data.
One way to check the model is the `.score()` method,
which determines the average accuracy (_Accuracy_) of the given test data in classification problems.

Further information on the `score()` method can be found at the following link:
http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.score

In [ ]:
clf_dt.score(X_test, y_test)

<span style="color:blue">
 Put the result in the context of what we already know based on the arithmetic mean alone.
 Would you use this model in practice?
</span>

In [ ]:
print("Probability that the customer will churn", df['Churn'].mean() * 100, "%")
print("Probability that the customer will not churn", (1 - df['Churn'].mean()) * 100, "%")

### Task 4.2: Predicting the likelihood of churn for specific customers

After the decision tree has been trained and the accuracy (_Accuracy_) has been evaluated, the next step is to analyze
whether specific customers would churn.

    SeniorCitizen
    Partner
    Dependents
    Tenure
    PhoneService
    PaperlessBilling
    MonthlyCharges
    ...
    

<span style="color:blue"> (a) Parameterize the variables with values ​​of your choice and interpret the result.
</span>

In [ ]:
# Your adjustments:

SeniorCitizen = 0
Partner = 0
Dependents = 0
Tenure = 0
PhoneService = 0
PaperlessBilling = 0
MonthlyCharges = 0
...

The previously parameterized variables are merged into an array and the trained decision tree is used to predict whether a customer will churn. 

<span style="color:blue"> (b) Run the following code and interpret the input and output.
</span>

In [ ]:
customer = np.array(
    [SeniorCitizen, Partner, Dependents, Tenure, PhoneService, PaperlessBilling, MonthlyCharges]
).reshape(1, -1)
pred = clf_dt.predict(customer)

if pred[0] == 0:  # No -> 0
    print("The customer stays")
elif pred[0] == 1:  # Yes -> 1
    print("The customer moves away")   

In addition to predicting the binary variable `Churn`, the probabilities of belonging to one of these classes can also be predicted.
This is made possible by the method `clf_dt.predict_proba()`.

<span style="color:blue"> (c) Run the following code and interpret the input and output.
</span>

In [ ]:
pred_probability = clf_dt.predict_proba(customer)
print("Probability that the customer stays:" , pred_probability[0][0])
print("Probability that the customer leaves:" , pred_probability[0][1])

To see what the decision tree created by the algorithm looks like, the following code can be used.

In [ ]:
from sklearn import tree

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
tree.plot_tree(
    clf_dt,
    ax=ax,
    feature_names=df.drop(['Customer ID', 'LoyaltyID', 'Churn'], axis=1).columns
)
plt.show()

The boxes in the bottom row represent the leaves, the other boxes are the nodes of the tree.
In these boxes, the top row is the condition that is checked.
If the condition is true, the left path is chosen and if the condition is false, the right path is chosen.

- The line with the gini value indicates the distribution of churned customers versus customers who remained with the company at the respective node or leaf.
- The row with the `sample` value indicates how many rows of data (without test data) have reached the respective node or leaf.
- The line with the value value indicates how many passengers (excluding test data) in the respective node or leaf remained as customers (1st value) or churned (2nd value).
- In the leaves, the larger value indicates the prediction the algorithm makes for new data.

<span style="color:blue"> (d) Why are the customers of the test data not counted?
</span> 

<span style="color:blue"> (e) Was this type of decision tree to be expected?
 Are there any distinctions and predictions that are surprising? Why is this?
</span>

### Task 4.3: Calibration of the decision tree
Revise the code given in section 4.3 of the exercise sheet.

<span style="color:blue">
 How can the decision tree be better preset to achieve better accuracy?
 Choose three hyperparameter variants when initializing the decision tree
 (i.e. change the value `max_depth=3`, for example), train the decision tree and determine the score for each of the selected variants.
 What do you notice?
</span>

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License Agreement" style="border-width:0; display:inline" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a> &nbsp;&nbsp;&nbsp;&nbsp;This work by Marvin Kastner is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>.